# Audit Congress Trading Signal — Agrégateurs tiers (Quiver, Capitol Trades)

Pendant logique de `house_ptr_coverage_audit.ipynb`. On **caractérise et on valide** les agrégateurs
comme complément/fallback aux sources primaires — on ne télécharge rien en masse.

**Usage strictement interne** (clause EIGA : pas d'usage commercial de ces données).

> La notebook tourne **sans clé** (mode `OFFLINE` sur un échantillon de démonstration **synthétique**,
> clairement étiqueté, jamais présenté comme réel) **et avec clé** (`QUIVER_API_KEY`).

In [9]:
import io, os, json, time, re, logging
from pathlib import Path
from datetime import datetime
import urllib.request, urllib.error
import pandas as pd
from IPython.display import display

# --- Chargement automatique de la cle depuis .env (racine du projet) ---
def _load_env_file():
    for candidate in [Path(".env"), Path("../.env")]:
        if candidate.exists():
            for line in candidate.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip())
            return
_load_env_file()

# --- Parametres ---
OUT_DIR     = Path("out_aggregators_audit")
SAMPLE_DIR  = OUT_DIR / "samples"
OUT_DIR.mkdir(exist_ok=True); SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

OVERLAP_YEAR = 2024
RATE_LIMIT_S = 0.6

QUIVER_API_KEY = os.environ.get("QUIVER_API_KEY", "").strip()
QUIVER_URL = "https://api.quiverquant.com/beta/bulk/congresstrading"

OFFLINE = (QUIVER_API_KEY == "")

HOUSE_PTR_INDEX = Path("out_house_audit") / f"ptr_index_{OVERLAP_YEAR}.csv"

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("agg_audit")

print(f"MODE : {'OFFLINE (echantillon local de demonstration)' if OFFLINE else 'LIVE (API Quiver)'}")
print(f"OVERLAP_YEAR = {OVERLAP_YEAR}  |  Index PTR House disponible : {HOUSE_PTR_INDEX.exists()}")

MODE : LIVE (API Quiver)
OVERLAP_YEAR = 2024  |  Index PTR House disponible : True


In [10]:
def _atomic_write_json(path: Path, obj):
    tmp = path.with_suffix(path.suffix + ".part")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)

def _ensure_offline_fixture(path: Path):
    """Echantillon SYNTHETIQUE pour mode OFFLINE — DONNEES DE DEMO, NON REELLES."""
    if path.exists():
        return
    demo = [
        {"Name":"Mark Alford","Filed":"2024-03-31","Traded":"2024-03-16","Ticker":"NVDA","Transaction":"Purchase","Trade_Size_USD":"1001.0","Chamber":"Representatives","Party":"Republican","TickerType":"ST"},
        {"Name":"Mark Alford","Filed":"2024-04-02","Traded":"2024-03-18","Ticker":"MSFT","Transaction":"Sale","Trade_Size_USD":"15001.0","Chamber":"Representatives","Party":"Republican","TickerType":"ST"},
        {"Name":"Marjorie Taylor Greene","Filed":"2024-05-10","Traded":"2024-04-22","Ticker":"AAPL","Transaction":"Purchase","Trade_Size_USD":"1001.0","Chamber":"Representatives","Party":"Republican","TickerType":"ST"},
        {"Name":"Josh Gottheimer","Filed":"2024-06-05","Traded":"2024-05-20","Ticker":"AMZN","Transaction":"Purchase","Trade_Size_USD":"15001.0","Chamber":"Representatives","Party":"Democrat","TickerType":"ST"},
        {"Name":"Josh Gottheimer","Filed":"2024-07-01","Traded":"2024-06-14","Ticker":"GOOGL","Transaction":"Purchase","Trade_Size_USD":"1001.0","Chamber":"Representatives","Party":"Democrat","TickerType":"ST"},
        {"Name":"Mark Alford","Filed":"2024-08-01","Traded":"2024-07-10","Ticker":"TSLA","Transaction":"Purchase","Trade_Size_USD":"1001.0","Chamber":"Representatives","Party":"Republican","TickerType":"ST"},
        {"Name":"Marjorie Taylor Greene","Filed":"2024-09-02","Traded":"2024-08-15","Ticker":"AMD","Transaction":"Purchase","Trade_Size_USD":"50001.0","Chamber":"Representatives","Party":"Republican","TickerType":"ST"},
        {"Name":"Daniel Meuser","Filed":"2024-10-01","Traded":"2024-09-12","Ticker":"PLTR","Transaction":"Purchase","Trade_Size_USD":"1001.0","Chamber":"Representatives","Party":"Republican","TickerType":"ST"},
        {"Name":"John Boozman","Filed":"2024-04-14","Traded":"2024-03-19","Ticker":"NVDA","Transaction":"Purchase","Trade_Size_USD":"1001.0","Chamber":"Senate","Party":"Republican","TickerType":"ST"},
        {"Name":"Tommy Tuberville","Filed":"2024-05-01","Traded":"2024-04-10","Ticker":"XOM","Transaction":"Purchase","Trade_Size_USD":"15001.0","Chamber":"Senate","Party":"Republican","TickerType":"ST"},
        {"Name":"Josh Gottheimer","Filed":"2023-03-15","Traded":"2023-02-20","Ticker":"DIS","Transaction":"Sale","Trade_Size_USD":"1001.0","Chamber":"Representatives","Party":"Democrat","TickerType":"ST"},
        {"Name":"Tommy Tuberville","Filed":"2023-08-20","Traded":"2023-07-29","Ticker":"AAPL","Transaction":"Purchase","Trade_Size_USD":"1001.0","Chamber":"Senate","Party":"Republican","TickerType":"ST"},
    ]
    _atomic_write_json(path, demo)
    log.warning("Fixture OFFLINE ecrite : %s (DONNEES NON REELLES)", path)

def fetch_quiver():
    cache = SAMPLE_DIR / ("quiver_OFFLINE_FIXTURE.json" if OFFLINE else "quiver_cache.json")
    if OFFLINE:
        _ensure_offline_fixture(cache)
        return json.loads(cache.read_text(encoding="utf-8"))
    if cache.exists() and cache.stat().st_size > 0:
        log.info("Quiver : cache present (%d octets)", cache.stat().st_size)
        return json.loads(cache.read_text(encoding="utf-8"))
    # User-Agent requis — urllib par defaut est bloque par l'API Quiver
    req = urllib.request.Request(QUIVER_URL, headers={
        "Authorization": f"Token {QUIVER_API_KEY}",
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0",
    })
    for attempt in range(3):
        try:
            time.sleep(RATE_LIMIT_S)
            with urllib.request.urlopen(req, timeout=120) as r:
                data = json.loads(r.read().decode("utf-8"))
            _atomic_write_json(cache, data)
            log.info("Quiver : %d records recuperes", len(data))
            return data
        except urllib.error.URLError as e:
            log.warning("Quiver tentative %d echouee : %s", attempt + 1, e)
            time.sleep(2 * (attempt + 1))
    raise RuntimeError("Echec Quiver (verifie cle / endpoint / reseau).")

raw_quiver = fetch_quiver()
print(f"{len(raw_quiver)} records bruts ({'fixture OFFLINE' if OFFLINE else 'API Quiver LIVE'}).")
display(pd.DataFrame(raw_quiver).head())

INFO Quiver : cache present (62853801 octets)


113673 records bruts (API Quiver LIVE).


,Ticker,TickerType,Company,Traded,Transaction,Trade_Size_USD,Status,Subholding,Description,Name,BioGuideID,Filed,Party,District,Chamber,Comments,Quiver_Upload_Time,excess_return,State,last_modified
0,AMZN,ST,None,2026-06-16,Sale,1001.0,None,None,NaN,Matthew Robert Van Epps,V000139,2026-06-17,Republican,7.0,Representatives,None,None,-4.84074146105724,None,2026-06-17
1,MSFT,ST,None,2026-06-16,Sale,1001.0,None,None,NaN,Matthew Robert Van Epps,V000139,2026-06-17,Republican,7.0,Representatives,None,None,-6.19707520581814,None,2026-06-17
2,GE,ST,None,2026-06-16,Sale,1001.0,None,None,NaN,Matthew Robert Van Epps,V000139,2026-06-17,Republican,7.0,Representatives,None,None,1.49298470408375,None,2026-06-17
3,TPR,ST,None,2026-06-16,Sale,15001.0,None,None,NaN,Matthew Robert Van Epps,V000139,2026-06-17,Republican,7.0,Representatives,None,None,0.401993357013215,None,2026-06-17
4,GEV,ST,None,2026-06-16,Sale,1001.0,None,None,NaN,Matthew Robert Van Epps,V000139,2026-06-17,Republican,7.0,Representatives,None,None,15.3141316841011,None,2026-06-17


In [11]:
# Champs API reels : Name, Traded, Filed, Trade_Size_USD, Chamber (vs ancien Representative, TransactionDate, ReportDate, Range, House)
NA = "non trouve"

def norm_chamber(v):
    v = (v or "").lower()
    if "senate" in v:                          return "Senate"
    if "house" in v or "representative" in v:  return "House"
    return NA

def norm_op(v):
    v = (v or "").strip().lower()
    if v.startswith("purchase"): return "Purchase"
    if "partial" in v:           return "Partial Sale"
    if "sale" in v:              return "Sale"
    if "exchange" in v:          return "Exchange"
    return v or NA

def norm_date(v):
    v = (v or "").strip()
    for fmt in ("%Y-%m-%d", "%m/%d/%Y"):
        try: return datetime.strptime(v, fmt).date().isoformat()
        except ValueError: pass
    return None

def normalize_quiver(raw):
    rows = []
    for d in raw:
        usd = d.get("Trade_Size_USD")
        try:    usd_val = float(usd) if usd is not None else None
        except: usd_val = None
        rows.append({
            "declarant_name":   d.get("Name") or NA,
            "chamber":          norm_chamber(d.get("Chamber")),
            "party":            d.get("Party") or NA,
            "transaction_date": norm_date(d.get("Traded")),
            "disclosure_date":  norm_date(d.get("Filed")),
            "ticker":           (d.get("Ticker") or NA).upper(),
            "operation_type":   norm_op(d.get("Transaction")),
            "amount_usd":       usd_val,
            "asset_type":       d.get("TickerType") or NA,
            "state":            d.get("State") or NA,
            "bioguide_id":      d.get("BioGuideID") or NA,
            "source":           "quiver",
        })
    return pd.DataFrame(rows)

qv = normalize_quiver(raw_quiver)
print(f"{len(qv)} records normalises.")
display(qv.head())

113673 records normalises.


,declarant_name,chamber,party,transaction_date,disclosure_date,ticker,operation_type,amount_usd,asset_type,state,bioguide_id,source
0,Matthew Robert Van Epps,House,Republican,2026-06-16,2026-06-17,AMZN,Sale,1001.0,ST,non trouve,V000139,quiver
1,Matthew Robert Van Epps,House,Republican,2026-06-16,2026-06-17,MSFT,Sale,1001.0,ST,non trouve,V000139,quiver
2,Matthew Robert Van Epps,House,Republican,2026-06-16,2026-06-17,GE,Sale,1001.0,ST,non trouve,V000139,quiver
3,Matthew Robert Van Epps,House,Republican,2026-06-16,2026-06-17,TPR,Sale,15001.0,ST,non trouve,V000139,quiver
4,Matthew Robert Van Epps,House,Republican,2026-06-16,2026-06-17,GEV,Sale,1001.0,ST,non trouve,V000139,quiver


In [12]:
def year_of(row):
    return (row["transaction_date"] or row["disclosure_date"] or "")[:4]

qv["year"] = qv.apply(year_of, axis=1)
yrs = qv["year"].replace("", pd.NA).dropna()

cov = {
    "source": "quiver",
    "audit_mode": "OFFLINE (demo)" if OFFLINE else "LIVE",
    "premiere_annee": yrs.min() if len(yrs) else "n/a",
    "derniere_annee": yrs.max() if len(yrs) else "n/a",
    "n_records": len(qv),
    "n_tickers_distincts": int(qv.loc[qv["ticker"] != "non trouve", "ticker"].nunique()),
    "n_declarants": int(qv["declarant_name"].nunique()),
    "n_house": int((qv["chamber"] == "House").sum()),
    "n_senate": int((qv["chamber"] == "Senate").sum()),
}
coverage_summary = pd.DataFrame([cov])
coverage_summary.to_csv(OUT_DIR / "coverage_summary.csv", index=False)

print(f"Tickers distincts MESURES : {cov['n_tickers_distincts']}"
      f"  (annonce par Quiver : ~1 800 -> a confronter sur un vrai pull LIVE)")
if OFFLINE:
    print("  /!\\ OFFLINE : chiffres sur echantillon de demo, sans valeur analytique.")
display(coverage_summary)
print("Volume par annee :");        display(qv["year"].value_counts().sort_index())
print("Repartition par chambre :"); display(qv["chamber"].value_counts())
print("Repartition par parti :");   display(qv["party"].value_counts())

Tickers distincts MESURES : 5121  (annonce par Quiver : ~1 800 -> a confronter sur un vrai pull LIVE)


,source,audit_mode,premiere_annee,derniere_annee,n_records,n_tickers_distincts,n_declarants,n_house,n_senate
0,quiver,LIVE,2012,2026,113673,5121,414,100331,13342


Volume par annee :


year
2012        9
2013      415
2014     5220
2015     5921
2016     7149
2017     8771
2018    11516
2019    11142
2020    12017
2021     7252
2022     9861
2023     9526
2024     7134
2025    13140
2026     4600
Name: count, dtype: int64

Repartition par chambre :


chamber
House     100331
Senate     13342
Name: count, dtype: int64

Repartition par parti :


party
Republican     57184
Democratic     56388
Independent       82
Libertarian       19
Name: count, dtype: int64

In [13]:
def missing_mask(col):
    return qv[col].isna() | (qv[col].astype(str).isin([NA, "", "None"]))

field_rows = []
for c in ["declarant_name","chamber","party","transaction_date","disclosure_date",
          "ticker","operation_type","amount_usd","asset_type","state"]:
    m = missing_mask(c)
    field_rows.append({"champ": c, "n_manquant": int(m.sum()),
                       "taux_manquant_%": round(100*m.mean(), 1)})
field_completeness = pd.DataFrame(field_rows)
field_completeness.to_csv(OUT_DIR / "field_completeness.csv", index=False)
display(field_completeness)

both = qv["transaction_date"].notna() & qv["disclosure_date"].notna()
print(f"\nLes deux dates presentes : {both.mean()*100:.0f}% des records")
if both.any():
    gap = (pd.to_datetime(qv.loc[both,"disclosure_date"]) -
           pd.to_datetime(qv.loc[both,"transaction_date"])).dt.days
    valid_gap = gap[gap >= 0]
    print(f"disclosure >= transaction : {(gap >= 0).mean()*100:.0f}% des records")
    if len(valid_gap):
        print(f"Delai disclosure-transaction (jours) : mediane {valid_gap.median():.0f}, "
              f"max {valid_gap.max():.0f}  (PTR officiel : <= 45 j)")

,champ,n_manquant,taux_manquant_%
0,declarant_name,0,0.0
1,chamber,0,0.0
2,party,0,0.0
3,transaction_date,0,0.0
4,disclosure_date,0,0.0
5,ticker,0,0.0
6,operation_type,0,0.0
7,amount_usd,12,0.0
8,asset_type,68282,60.1
9,state,113673,100.0



Les deux dates presentes : 100% des records
disclosure >= transaction : 100% des records
Delai disclosure-transaction (jours) : mediane 28, max 4112  (PTR officiel : <= 45 j)


In [14]:
## Validation croisee niveau DECLARANTS
## Source officielle : index PTR House reel (ptr_index_OVERLAP_YEAR.csv)
## Trade-level impossible ici — les transactions sont dans les PDFs (prevu J3-4)

def name_key(s):
    s = re.sub(r"\(.*?\)", "", s or "")
    s = re.sub(r"\b(hon|mr|mrs|ms|dr|sen|rep)\.?\b", "", s, flags=re.I)
    toks = re.sub(r"[^a-zA-Z ]", " ", s).split()
    return " ".join(sorted(t.upper() for t in toks))

# --- Cote Quiver ---
agg_h = qv[(qv["chamber"] == "House") & (qv["year"] == str(OVERLAP_YEAR))].copy()
agg_h["name_key"] = agg_h["declarant_name"].map(name_key)
quiver_names = set(agg_h["name_key"].unique())

# --- Cote officiel : index PTR reel ---
if not HOUSE_PTR_INDEX.exists():
    log.warning("Index PTR manquant — lancer house_ptr_coverage_audit.ipynb d'abord (%s)", HOUSE_PTR_INDEX)
    official_names = set()
    source_label = "MANQUANT (lancer house_ptr_coverage_audit.ipynb)"
else:
    ptr_off = pd.read_csv(HOUSE_PTR_INDEX)
    ptr_off["name_key"] = (ptr_off["first"].fillna("") + " " + ptr_off["last"].fillna("")).str.strip().map(name_key)
    official_names = set(ptr_off["name_key"].unique())
    source_label = f"ptr_index_{OVERLAP_YEAR}.csv ({len(ptr_off)} depots, {len(official_names)} declarants)"

# --- Appariement ---
matched_names = quiver_names & official_names
only_quiver   = quiver_names - official_names
only_official = official_names - quiver_names
taux = round(100 * len(matched_names) / len(quiver_names), 1) if quiver_names else 0.0

print(f"Quiver House {OVERLAP_YEAR} : {len(agg_h)} transactions | {len(quiver_names)} declarants")
print(f"Index officiel : {source_label}")
if quiver_names:
    print(f"\nDeclarants Quiver dans l'officiel : {len(matched_names)}/{len(quiver_names)} ({taux:.0f}%)")
if only_quiver:
    print(f"  Seulement dans Quiver (a verifier) : {sorted(only_quiver)}")
if OFFLINE:
    print("\n  /!\\ OFFLINE : noms issus de la fixture de demo — couverture non representative.")

crossval = pd.DataFrame([{
    "overlap_year": OVERLAP_YEAR,
    "niveau_validation": "declarants (trade-level prevu J3-4 apres parsing PDF)",
    "n_declarants_quiver": len(quiver_names),
    "n_declarants_officiels": len(official_names),
    "declarants_apparies": len(matched_names),
    "seulement_quiver": len(only_quiver),
    "seulement_officiel": len(only_official),
    "taux_couverture_%": taux,
    "source_officielle": source_label,
    "audit_mode": "OFFLINE (demo)" if OFFLINE else "LIVE",
}])
crossval.to_csv(OUT_DIR / "crossval_declarants.csv", index=False)
display(crossval)
print("-> crossval_declarants.csv")

Quiver House 2024 : 6550 transactions | 76 declarants
Index officiel : ptr_index_2024.csv (451 depots, 98 declarants)

Declarants Quiver dans l'officiel : 60/76 (79%)
  Seulement dans Quiver (a verifier) : ['AUGUST II LEE PFLUGER', 'AUSTIN SCOTT', 'CALHOUN JAMES', 'CASE ED', 'DONALD NORCROSS W', 'FRANKLIN SCOTT SCOTT', 'GUY RESCHENTHALER', 'H JR KEAN THOMAS', 'HAROLD ROGERS', 'HOYLE VALERIE', 'III RUDY YAKYM', 'JOHN RITCHIE TORRES', 'JOHNSON JULIE', 'JULIA LETLOW', 'KHANNA RO', 'LISA MCCLAIN']


,overlap_year,niveau_validation,n_declarants_quiver,n_declarants_officiels,declarants_apparies,seulement_quiver,seulement_officiel,taux_couverture_%,source_officielle,audit_mode
0,2024,declarants (trade-level prevu J3-4 apres parsi...,76,98,60,16,38,78.9,"ptr_index_2024.csv (451 depots, 98 declarants)",LIVE


-> crossval_declarants.csv


In [15]:
ech = qv.sample(min(6, len(qv)), random_state=0)[
    ["declarant_name","chamber","ticker","operation_type","transaction_date","disclosure_date","amount_usd"]
].reset_index(drop=True)
ech["a_verifier_sur"] = "Capitol Trades -> rechercher : " + ech["declarant_name"]

print("A VERIFIER A LA MAIN (confirmer ticker + LES DEUX DATES + montant) :")
print("Capitol Trades : https://www.capitoltrades.com")
display(ech)

A VERIFIER A LA MAIN (confirmer ticker + LES DEUX DATES + montant) :
Capitol Trades : https://www.capitoltrades.com


,declarant_name,chamber,ticker,operation_type,transaction_date,disclosure_date,amount_usd,a_verifier_sur
0,Ro Khanna,House,KMB,Purchase,2026-01-21,2026-02-06,1001.0,Capitol Trades -> rechercher : Ro Khanna
1,James R. Langevin,House,DIA,Purchase,2019-05-31,2019-06-29,1001.0,Capitol Trades -> rechercher : James R. Langevin
2,Bill Flores,House,MMM,Sale,2016-04-18,2016-05-30,1001.0,Capitol Trades -> rechercher : Bill Flores
3,Donald Sternoff Beyer Jr.,House,NVDA,Sale,2022-02-28,2022-03-03,15001.0,Capitol Trades -> rechercher : Donald Sternoff...
4,Ro Khanna,House,DRE,Purchase,2021-05-28,2021-10-14,1001.0,Capitol Trades -> rechercher : Ro Khanna
5,Michael T. McCaul,House,BMY,Sale,2016-08-10,2016-09-23,1001.0,Capitol Trades -> rechercher : Michael T. McCaul


In [16]:
taux = float(crossval["taux_couverture_%"].iloc[0])
verdict = pd.DataFrame([
    {"question": "Fallback recent (>= 2016) ?",
     "reponse": f"Couverture declarants House mesuree : {taux}% sur {OVERLAP_YEAR}. "
                f"Trade-level apres J3-4 (parsing PDF)."},
    {"question": "Fallback historique Senat 2013-2015 ?",
     "reponse": "NON — Quiver demarre en janv. 2016, il ne comble pas ce trou."},
    {"question": "Fallback Senat 2016-2019 ?",
     "reponse": "A trancher sur un vrai pull LIVE (couverture Senate mesuree en section 3)."},
    {"question": "Capitol Trades ?",
     "reponse": "Verification manuelle seulement (pas d'API publique, ~3 ans d'historique)."},
])
display(verdict)

manifest = pd.DataFrame([{
    "source": "quiver",
    "audit_mode": "OFFLINE (demo)" if OFFLINE else "LIVE",
    "ok": len(qv) > 0,
    "n_records": len(qv),
    "annees": f"{cov['premiere_annee']}-{cov['derniere_annee']}",
    "tickers_distincts_mesures": cov["n_tickers_distincts"],
    "taux_couverture_declarants_%": taux,
    "niveau_validation": "declarants (trade-level prevu J3-4)",
}])
manifest.to_csv(OUT_DIR / "manifest.csv", index=False)
display(manifest)
print()
print("Sorties dans", OUT_DIR, ":")
for p in sorted(OUT_DIR.glob("*.csv")): print("  -", p.name)

,question,reponse
0,Fallback recent (>= 2016) ?,Couverture declarants House mesuree : 78.9% su...
1,Fallback historique Senat 2013-2015 ?,"NON — Quiver demarre en janv. 2016, il ne comb..."
2,Fallback Senat 2016-2019 ?,A trancher sur un vrai pull LIVE (couverture S...
3,Capitol Trades ?,Verification manuelle seulement (pas d'API pub...


,source,audit_mode,ok,n_records,annees,tickers_distincts_mesures,taux_couverture_declarants_%,niveau_validation
0,quiver,LIVE,True,113673,2012-2026,5121,78.9,declarants (trade-level prevu J3-4)



Sorties dans out_aggregators_audit :
  - coverage_summary.csv
  - crossval_declarants.csv
  - field_completeness.csv
  - manifest.csv


## Ce qui reste pour finaliser l'audit agrégateurs

**Prochaine action bloquante — obtenir la clé Quiver :**
- S'inscrire sur [quiverquant.com](https://www.quiverquant.com) (plan gratuit : 100 req/jour)
- `export QUIVER_API_KEY="ta_cle"` dans le terminal → relancer le notebook en LIVE
- Le `coverage_summary.csv` contiendra alors les vrais métriques (annonce : ~1 800 tickers)

**Ce que la cross-validation peut mesurer à ce stade (J1-2) :**
- Niveau **déclarants** : % des noms Quiver retrouvés dans l'index PTR officiel — déjà branché sur `ptr_index_2024.csv` (vraies données)
- Niveau **trade** : impossible sans parsing PDF — prévu J3-4

**Pour la décision d'architecture (livrable J1-2) :**
- `coverage_summary.csv` → fenêtre temporelle réelle + couverture Senate vs House
- `crossval_declarants.csv` → taux de couverture des déclarants Quiver vs officiel
- 2-3 records vérifiés manuellement sur [Capitol Trades](https://www.capitoltrades.com) (les deux dates)

**Sorties valides uniquement en mode LIVE** — en OFFLINE, `audit_mode = "OFFLINE (demo)"` dans chaque CSV.